In [1]:
import pandas as pd                                     # pandas: tables
from scipy import stats                                 # scipy.stats: chi-square test
from statsmodels.stats.power import NormalIndPower      # sample-size calculator for yes/no outcomes
from statsmodels.stats.proportion import proportion_effectsize   # turns two percents into an effect size

In [2]:
url = "https://raw.githubusercontent.com/haijinwootest/email_data_analysis/main/Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"
df = pd.read_csv(url)                     # read the file into a table called df
print(df.shape)                           # 64,000 rows, 12 columns
print(df[["segment", "visit"]].head())    # the first 5 rows of the two columns we need

(64000, 12)
         segment  visit
0  Womens E-Mail      0
1      No E-Mail      0
2  Womens E-Mail      0
3    Mens E-Mail      0
4  Womens E-Mail      0


In [3]:
print(df.isna().sum().sum())    # total number of empty cells in the whole table

0


## The question and the hypotheses
Question: do the three groups have different percents of customers who visit the website?
With three groups we answer it two groups at a time. That gives three pairs:
1.  Mens E-Mail against No E-Mail
2.  Womens E-Mail against No E-Mail
3.  Mens E-Mail against Womens E-Mail
There are three groups, but each test looks 

• Null hypothesis: the two groups in this pair visit equally. Any difference is chance. <br>
• Alternative hypothesis: the two groups in this pair visit differently. <br>
<br>
  Example for pair 1: "Customers who got the men's email and customers who got no email visit equally" (null) against "they visit differently" (alternative).

the stricter line. Three tests means three chances of a false win, so the line is 0.05 ÷ 3 = 0.0167 

Was the sample big enough?
1.  Starting value: from past records, the shop knows that without an email about 11% of customers visit the website in two weeks. This is the value expected for the No E-Mail group, which is the comparison group (the baseline): each email group is compared against it.
2.  Smallest change worth catching: the shop says a rise of 2 points (from 11% to 13%) pays for the emails.
3.  The line: for a planned test with three pairs, use the stricter line here too: 0.0167.
4.  Power 80%.

In [5]:
needed = NormalIndPower().solve_power(
    effect_size=proportion_effectsize(0.11, 0.13),   # from 11% to 13%
    alpha=0.05 / 3,                                  # the stricter line, because of 3 pairs
    power=0.8                                        # 80% power
)
print(needed)    # customers needed in EACH group

5518.772959285544


Each group has about 21,300 customers, so the sample is big enough.

In [11]:
count=df["segment"].value_counts()
print(count)

segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64


In [13]:
p=count/count.sum()  * 100
print(p.round(2))

segment
Womens E-Mail    33.42
Mens E-Mail      33.29
No E-Mail        33.29
Name: count, dtype: float64


In [15]:
print(stats.chisquare(count).pvalue)

0.9036929575604196


 Above 0.05, so the split is normal chance

In [16]:
table=pd.crosstab(df["segment"],df["visit"])
print(table)

visit              0     1
segment                   
Mens E-Mail    17413  3894
No E-Mail      19044  2262
Womens E-Mail  18149  3238


In [17]:
p=table[1]/table.sum(axis=1) * 100

print(p.round(1))


segment
Mens E-Mail      18.3
No E-Mail        10.6
Womens E-Mail    15.1
dtype: float64


In [21]:
# For group 1 Men's Email versus no Email 
mens_none = df.loc[(df["segment"]=="Mens E-Mail") | (df["segment"]=="No E-Mail"), :]
table1=pd.crosstab(mens_none["segment"],mens_none["visit"])

p1=stats.chi2_contingency(table1).pvalue
print("Men's vs No email",p1)

Men's vs No email 7.754092221027678e-112


In [20]:
# Pair 2: Womens E-Mail against No E-Mail
womens_none = df.loc[(df["segment"] == "Womens E-Mail") | (df["segment"] == "No E-Mail"), :]
table_2 = pd.crosstab(womens_none["segment"], womens_none["visit"])
p_2 = stats.chi2_contingency(table_2).pvalue
print("Womens vs No email:", p_2)

Womens vs No email: 3.89649335937488e-44


In [22]:
# Pair 3: Mens E-Mail against Womens E-Mail
mens_womens = df.loc[(df["segment"] == "Mens E-Mail") | (df["segment"] == "Womens E-Mail"), :]
table_3 = pd.crosstab(mens_womens["segment"], mens_womens["visit"])
p_3 = stats.chi2_contingency(table_3).pvalue
print("Mens vs Womens:", p_3)

Mens vs Womens: 4.261498076023709e-18


All three differences are real.

Both emails are above the 2 points the shop wanted. The men's email is clearly the strongest.

Cramér's V: the standard effect size for a chi-square test

In [24]:
from scipy.stats.contingency import association     # the tool that calculates Cramér's V

v_1 = association(table1, method="cramer")         # input: the count table of the pair
v_2 = association(table_2, method="cramer")
v_3 = association(table_3, method="cramer")
print(round(v_1, 3), round(v_2, 3), round(v_3, 3))

0.109 0.068 0.042


Why "small" but still important? Cramér's V is small because most customers do not visit in any group (82% to 89% did not visit). But for the shop, going from 10.62% to 18.28% means about 72% more visitors (18.28 ÷ 10.62 = 1.72) from one email. So report both: Cramér's V for the statistics, and the difference in percent for the business decision.